# CatBoost economic-block ablation

This notebook retrains the 15-basis-point, 24-hour CatBoost specification recorded in the full-retraining run, then removes one economic feature block at a time. It reports row-level discrimination and held-out operational warning metrics.

## Comparison design

- The reference run's saved model parameters, preprocessing choice, label definition, and chronological split sizes define the experiment.
- The full-feature model and every ablation are retrained on the same training rows. Robust scaling is fit on training rows only.
- Each model uses validation for early stopping and alert-threshold selection under the saved false-alert budget. The final 30% remains held out for every test metric.
- Economic blocks follow the group definitions in `9. SHAP explanations of Early Warning Model/SHAP_details.ipynb`. Any unassigned model inputs form an explicit Other / unclassified block.
- Utility is reported both gross per event and net of the run's false-alert cost. Event recall is the timely, declustered recall used by the operational evaluator.

Run all cells from this folder or the repository root. The notebook writes a CSV and run metadata under lightning_logs/ when the model fits complete.

In [ ]:
import json
import sys
import time
import joblib
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from catboost import CatBoostClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
)

# Resolve the cross-validation folder from the notebook folder, repository
# root, or its parent directory.
cwd = Path.cwd().resolve()
notebook_dir_candidates = [
    cwd,
    cwd / "8. Early-Warning Classifier Cross Validation",
    cwd.parent / "8. Early-Warning Classifier Cross Validation",
]
NOTEBOOK_DIR = next(
    (path for path in notebook_dir_candidates if (path / "cv_model_comparison.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Could not locate the cross-validation folder.")
REPO_ROOT = NOTEBOOK_DIR.parent
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from early_warning_evaluation import choose_threshold_by_utility, evaluate_early_warning

EXPERIMENT_NAME = "cv_model_comparison_2026-09-23_15bp_full_retraining"
RUN_GLOB = "catboost_threshold_15_alpha_0.3_fullfeatures_*"
EXPERIMENT_DIR = NOTEBOOK_DIR / "lightning_logs" / EXPERIMENT_NAME
run_candidates = sorted(EXPERIMENT_DIR.glob(RUN_GLOB))
if len(run_candidates) != 1:
    raise FileNotFoundError(
        f"Expected one run matching {RUN_GLOB!r} in {EXPERIMENT_DIR}; "
        f"found {len(run_candidates)}."
    )
RUN_DIR = run_candidates[0]

with (RUN_DIR / "hparams.json").open(encoding="utf-8") as file:
    hparams = json.load(file)
with (RUN_DIR / "artifacts" / "models" / "signature.json").open(encoding="utf-8") as file:
    signature = json.load(file)
with (RUN_DIR / "artifacts" / "reports" / "operating_threshold.json").open(encoding="utf-8") as file:
    saved_operating_threshold = json.load(file)
with (RUN_DIR / "metrics.json").open(encoding="utf-8") as file:
    saved_run_metrics = json.load(file)

if (
    hparams["model_name"] != "catboost"
    or int(hparams["target_threshold"]) != 15
    or float(hparams["alpha"]) != 0.3
    or int(hparams["target_window"]) != 24
):
    raise ValueError("The selected run is not the expected 15-bp, alpha=0.3 CatBoost run.")

# Resolve the relative dataset path stored in hparams.json.
dataset_relative = Path(hparams["dataset_path"])
dataset_candidates = [
    dataset_relative if dataset_relative.is_absolute() else cwd / dataset_relative,
    NOTEBOOK_DIR / dataset_relative,
    REPO_ROOT / dataset_relative,
    REPO_ROOT
    / "9. SHAP explanations of Early Warning Model"
    / "preprocessed_datasets"
    / dataset_relative.name,
]
DATASET_PATH = next((path.resolve() for path in dataset_candidates if path.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not locate the dataset recorded in hparams.json. Checked:\n"
        + "\n".join(str(path) for path in dataset_candidates)
    )

TIME_COL = "timestamp"
TARGET_COL = "target"
df = pd.read_parquet(DATASET_PATH)

# Match full-retraining preparation: timestamp comes from the parquet index,
# then seven lagged peg-deviation features are rebuilt.
df[TIME_COL] = df.index
for lag in range(1, 8):
    df[f"depeg_bps_lag{lag}h"] = df["depeg_bps"].shift(lag)
df = df.dropna().copy()
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)

if TARGET_COL not in df:
    raise KeyError(f"Dataset is missing {TARGET_COL!r}.")
FEATURES = list(signature["input_columns"])
dataset_features = [column for column in df.columns if column not in [TIME_COL, TARGET_COL]]
if dataset_features != FEATURES:
    raise ValueError(
        "Dataset features/order do not match the saved model signature. "
        "Use the same dataset version as the full-retraining run."
    )
non_numeric = df[FEATURES].select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
if non_numeric:
    raise TypeError(f"Non-numeric model inputs found: {non_numeric}")

# Recreate chronological train / validation / test partitions. The test length
# follows the int(fraction * rows) convention in the companion SHAP notebook.
TEST_FRAC = float(hparams["test_size"])
VAL_FRAC = float(hparams["val_size"])
test_n = int(TEST_FRAC * len(df))
val_n = int(VAL_FRAC * len(df))
test_start = len(df) - test_n
val_start = test_start - val_n
if val_start <= 0 or val_n <= 0 or test_n <= 0:
    raise ValueError("Recorded train / validation / test fractions leave an empty split.")

train_frame = df.iloc[:val_start].reset_index(drop=True)
validation_frame = df.iloc[val_start:test_start].reset_index(drop=True)
test_frame = df.iloc[test_start:].reset_index(drop=True)

X_train_raw = train_frame[FEATURES].copy()
y_train = train_frame[TARGET_COL].astype(int).copy()
X_validation_raw = validation_frame[FEATURES].copy()
y_validation = validation_frame[TARGET_COL].astype(int).copy()
X_test_raw = test_frame[FEATURES].copy()
y_test = test_frame[TARGET_COL].astype(int).copy()

# Verify that the recreated holdout exactly matches the saved run's test rows.
saved_predictions_path = RUN_DIR / "artifacts" / "predictions" / "test_pred_proba.parquet"
saved_test_predictions = pd.read_parquet(saved_predictions_path)
expected_timestamps = pd.to_datetime(test_frame[TIME_COL], utc=True).to_numpy()
saved_timestamps = pd.to_datetime(saved_test_predictions["timestamp"], utc=True).to_numpy()
if len(saved_test_predictions) != len(test_frame) or not np.array_equal(
    expected_timestamps, saved_timestamps
):
    raise ValueError("Recreated test rows do not match saved full-retraining predictions.")

# Use CatBoost-specific parameters from the saved estimator, with run hparams
# supplying the principal values logged by the retraining workflow.
template_model = joblib.load(RUN_DIR / "artifacts" / "models" / "model.joblib")
CATBOOST_PARAMS = dict(template_model.get_params())
CATBOOST_PARAMS.update(
    {
        "iterations": int(hparams["n_estimators"]),
        "learning_rate": float(hparams["learning_rate"]),
        "depth": int(hparams["max_depth"]),
        "scale_pos_weight": float(hparams["scale_pos_weight_used"]),
        "verbose": False,
        "allow_writing_files": False,
    }
)
CATBOOST_PARAMS.setdefault("loss_function", "Logloss")
CATBOOST_PARAMS.setdefault("eval_metric", hparams.get("eval_metric", "AUC"))
CATBOOST_PARAMS.setdefault("random_seed", 1233)
CATBOOST_PARAMS.setdefault("od_type", "Iter")
CATBOOST_PARAMS.setdefault("od_wait", int(hparams["early_stopping_rounds"]))

if hparams["scaler"] != "robust":
    raise ValueError(f"Expected RobustScaler in reference run, found {hparams['scaler']!r}.")

print(f"Reference run: {RUN_DIR}")
print(f"Dataset: {DATASET_PATH}")
print(
    f"Rows train/validation/test: {len(train_frame):,} / "
    f"{len(validation_frame):,} / {len(test_frame):,}; features: {len(FEATURES)}"
)
print(
    f"Positive rate train/validation/test: {y_train.mean():.3%} / "
    f"{y_validation.mean():.3%} / {y_test.mean():.3%}"
)
print("CatBoost parameters used for every fit:")
display(pd.Series(CATBOOST_PARAMS, name="value").to_frame())
print(
    "Saved-run reference: "
    f"AUC={saved_run_metrics.get('test_roc_auc', np.nan):.4f}, "
    f"AUPRC={saved_run_metrics.get('test_auprc', np.nan):.4f}, "
    f"event recall={saved_run_metrics.get('test_operating_timely_event_recall', np.nan):.3f}, "
    f"net utility/event={saved_run_metrics.get('test_operating_event_utility_score', np.nan):.3f}, "
    f"saved threshold={saved_operating_threshold.get('operating_threshold', np.nan):.6f}"
)

## Define the economic blocks

The memberships follow the existing SHAP grouping. Assignment is made in order, so every predictor belongs to one and only one block. This keeps the ablation units consistent with the repository's grouped SHAP analysis.

In [ ]:
def has_prefix(feature, prefixes):
    return any(feature.startswith(prefix) for prefix in prefixes)


candidate_blocks = OrderedDict(
    [
        (
            "Uniswap V3 liquidity-curve shape",
            [
                feature
                for feature in FEATURES
                if (
                    feature.startswith("Gegenbauer_")
                    or feature.startswith("E_")
                    or "ratio" in feature
                    or feature
                    in [
                        "tangent_up",
                        "tangent_down",
                        "swap_size_imbalance",
                        "tvlUSD_100",
                        "tvlUSD_500",
                    ]
                )
            ],
        ),
        (
            "Curve 3pool liquidity conditions",
            [
                feature
                for feature in [
                    "w_USDC",
                    "w_USDT",
                    "curve_entropy",
                    "gauge_share_3crv",
                    "totalValueLockedUSD",
                ]
                if feature in FEATURES
            ],
        ),
        (
            "Broader market conditions",
            [
                feature
                for feature in FEATURES
                if has_prefix(feature, ("eth_", "btc_"))
                or feature in ["usd_index", "fx_volatility", "fear_greed_index"]
            ],
        ),
        (
            "Historical peg deviation",
            [
                feature
                for feature in FEATURES
                if feature == "depeg_bps" or feature.startswith("depeg_bps_lag")
            ],
        ),
        (
            "Liquidity ownership and position structure",
            [
                feature
                for feature in [
                    "hhi_24h_rolling_mean",
                    "tick_width_24h_rolling_median",
                    "n_in_range_log_return",
                    "weighted_mean_age_hours",
                ]
                if feature in FEATURES
            ],
        ),
        (
            "Trading velocity and flows",
            [
                feature
                for feature in [
                    "swap_count_100",
                    "swap_count_500",
                    "net_amountUSD_100",
                    "net_amountUSD_500",
                    "net_amount0",
                    "hourlyVolumeUSD",
                ]
                if feature in FEATURES
            ],
        ),
        (
            "AAVE lending market conditions",
            [
                feature
                for feature in [
                    "supplied_USD_usdt",
                    "utilisation_rate_usdt",
                    "supplied_USD_usdc",
                    "utilisation_rate_usdc",
                    "liquidation_USD",
                ]
                if feature in FEATURES
            ],
        ),
    ]
)

FEATURE_BLOCKS = OrderedDict()
assigned = set()
for block_name, block_features in candidate_blocks.items():
    unique = [
        feature
        for feature in block_features
        if feature in FEATURES and feature not in assigned
    ]
    if unique:
        FEATURE_BLOCKS[block_name] = unique
        assigned.update(unique)

unclassified = [feature for feature in FEATURES if feature not in assigned]
if unclassified:
    FEATURE_BLOCKS["Other / unclassified"] = unclassified
    assigned.update(unclassified)

if assigned != set(FEATURES):
    raise AssertionError("Economic blocks must cover the complete saved model input signature.")
if sum(len(features) for features in FEATURE_BLOCKS.values()) != len(FEATURES):
    raise AssertionError("A feature was assigned to more than one block.")

block_summary = pd.DataFrame(
    [
        {
            "economic_block": name,
            "n_features": len(features),
            "features": ", ".join(features),
        }
        for name, features in FEATURE_BLOCKS.items()
    ]
)
display(block_summary[["economic_block", "n_features"]])
print(f"All {len(FEATURES)} input features are assigned exactly once.")

## Fit and evaluate the full model and ablations

The positive-class weight stays at the value recorded for the reference run. Since the rows and labels are identical, every feature set has the same training class balance. Each fit can select a different best iteration using the same validation early-stopping rule.

Each model gets its own operating threshold, selected on validation only under the saved false-alert budget. Test AUC and AUPRC are row-level metrics; event recall, alert burden, and utility use the repository's declustered operational evaluator.

In [ ]:
warning_window = hparams["warning_window_hours"]
FALSE_ALERT_BUDGET = float(hparams["false_alert_budget_per_month"])
THRESHOLD_GRID_SIZE = 201
EVALUATION_KWARGS = {
    "timestamp_col": TIME_COL,
    "depeg_col": "depeg_bps",
    "threshold_bps": float(hparams["target_threshold"]),
    "depeg_side": hparams["depeg_side"],
    "dynamic_threshold": bool(hparams["dynamic_threshold"]),
    "min_lead_hours": float(warning_window[0]),
    "max_lead_hours": float(warning_window[1]),
    "target_lead_hours": float(hparams["utility_target_lead_hours"]),
    "min_lead_utility": float(hparams["min_lead_utility"]),
    "cooldown_hours": float(hparams["alert_cooldown_hours"]),
    "false_alert_cost": float(hparams["false_alert_cost"]),
    "utility_power": float(hparams["utility_power"]),
    "event_reset_hours": float(hparams.get("depeg_event_reset_hours", 24.0)),
    "event_context_frame": df,
}


def safe_auc(labels, probabilities):
    if pd.Series(labels).nunique() < 2:
        return np.nan
    return float(roc_auc_score(labels, probabilities))


def safe_auprc(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    if labels.sum() == 0:
        return np.nan
    return float(average_precision_score(labels, probabilities))


def scaled_frame(scaler, values, columns, index):
    return pd.DataFrame(
        scaler.transform(values),
        columns=columns,
        index=index,
    )


def fit_and_score(feature_set_name, selected_features, removed_block):
    started = time.perf_counter()

    X_train = X_train_raw[selected_features]
    X_validation = X_validation_raw[selected_features]
    X_test = X_test_raw[selected_features]

    scaler = RobustScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=selected_features,
        index=X_train.index,
    )
    X_validation_scaled = scaled_frame(
        scaler, X_validation, selected_features, X_validation.index
    )
    X_test_scaled = scaled_frame(scaler, X_test, selected_features, X_test.index)

    model = CatBoostClassifier(**CATBOOST_PARAMS)
    if y_validation.nunique() > 1:
        model.fit(
            X_train_scaled,
            y_train,
            eval_set=(X_validation_scaled, y_validation),
            use_best_model=True,
            verbose=False,
        )
    else:
        model.fit(X_train_scaled, y_train, verbose=False)

    validation_probabilities = model.predict_proba(X_validation_scaled)[:, 1]
    operating_threshold, validation_metrics = choose_threshold_by_utility(
        validation_frame,
        validation_probabilities,
        false_alert_budget_per_month=FALSE_ALERT_BUDGET,
        threshold_grid_size=THRESHOLD_GRID_SIZE,
        **EVALUATION_KWARGS,
    )

    test_probabilities = model.predict_proba(X_test_scaled)[:, 1]
    test_metrics, _ = evaluate_early_warning(
        test_frame,
        test_probabilities,
        operating_threshold,
        **EVALUATION_KWARGS,
    )

    prevalence = float(y_test.mean())
    brier = float(brier_score_loss(y_test, test_probabilities))
    reference_brier = prevalence * (1.0 - prevalence)
    brier_skill = 1.0 - brier / reference_brier if reference_brier > 0 else np.nan
    row_alerts = (test_probabilities >= operating_threshold).astype(int)
    best_iteration = model.get_best_iteration()

    return {
        "feature_set": feature_set_name,
        "removed_block": removed_block,
        "n_features": len(selected_features),
        "best_iteration": int(best_iteration + 1) if best_iteration >= 0 else int(model.tree_count_),
        "operating_threshold": float(operating_threshold),
        "validation_event_recall": validation_metrics["timely_event_recall"],
        "validation_net_utility_per_event": validation_metrics["event_utility_score"],
        "validation_false_alerts_per_month": validation_metrics["false_alerts_per_month"],
        "test_auc": safe_auc(y_test, test_probabilities),
        "test_auprc": safe_auprc(y_test, test_probabilities),
        "test_brier_score": brier,
        "test_brier_skill_score": brier_skill,
        "test_row_precision_at_operating_threshold": float(
            precision_score(y_test, row_alerts, zero_division=0)
        ),
        "test_row_recall_at_operating_threshold": float(
            recall_score(y_test, row_alerts, zero_division=0)
        ),
        "test_row_f1_at_operating_threshold": float(
            f1_score(y_test, row_alerts, zero_division=0)
        ),
        "test_event_recall": test_metrics["timely_event_recall"],
        "test_n_events": test_metrics["n_events"],
        "test_events_detected": test_metrics["events_detected"],
        "test_median_lead_hours": test_metrics["median_lead_hours"],
        "test_alert_episodes": test_metrics["alert_episodes"],
        "test_false_alert_episodes": test_metrics["false_alert_episodes"],
        "test_false_alerts_per_month": test_metrics["false_alerts_per_month"],
        "test_alert_episode_precision": test_metrics["alert_episode_precision"],
        "test_gross_event_utility": test_metrics["event_utility"],
        "test_net_utility_per_event": test_metrics["operational_utility_per_event"],
        "fit_seconds": time.perf_counter() - started,
    }


feature_sets = [("All features", FEATURES.copy(), None)]
for block_name, removed_features in FEATURE_BLOCKS.items():
    retained = [feature for feature in FEATURES if feature not in removed_features]
    if not retained:
        raise ValueError(f"Removing {block_name!r} would leave no features.")
    feature_sets.append((f"Without: {block_name}", retained, block_name))

ablation_rows = []
for run_number, (name, selected, removed) in enumerate(feature_sets, start=1):
    print(f"[{run_number}/{len(feature_sets)}] {name}: {len(selected)} features")
    ablation_rows.append(fit_and_score(name, selected, removed))

results = pd.DataFrame(ablation_rows)
print("Ablation fits complete.")

## Held-out results

Delta columns compare each ablation with the retrained full-feature baseline. Negative deltas mean that removing the block lowered the metric; a positive delta means the ablation scored higher on this holdout.

In [ ]:
DELTA_METRICS = [
    "test_auc",
    "test_auprc",
    "test_event_recall",
    "test_net_utility_per_event",
    "test_false_alerts_per_month",
]
baseline = results.loc[results["feature_set"] == "All features"].iloc[0]
for metric in DELTA_METRICS:
    results[f"delta_{metric}"] = results[metric] - baseline[metric]

baseline_row = results[results["feature_set"] == "All features"]
ablation_only = results[results["feature_set"] != "All features"].sort_values(
    "test_net_utility_per_event",
    ascending=False,
    na_position="last",
)
results = pd.concat([baseline_row, ablation_only], ignore_index=True)

reported_columns = [
    "feature_set",
    "n_features",
    "best_iteration",
    "operating_threshold",
    "test_auc",
    "test_auprc",
    "test_brier_score",
    "test_event_recall",
    "test_median_lead_hours",
    "test_false_alerts_per_month",
    "test_alert_episode_precision",
    "test_gross_event_utility",
    "test_net_utility_per_event",
    "delta_test_auc",
    "delta_test_auprc",
    "delta_test_event_recall",
    "delta_test_net_utility_per_event",
    "delta_test_false_alerts_per_month",
]
display(
    results[reported_columns].style.format(
        {
            column: "{:.3f}"
            for column in reported_columns
            if column not in ["feature_set", "n_features", "best_iteration"]
        }
    )
)

OUTPUT_DIR = (
    NOTEBOOK_DIR
    / "lightning_logs"
    / "cv_model_comparison_2026-09-23_15bp_economic_block_ablation"
    / "artifacts"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUTPUT_DIR / "ablation_metrics.csv"
METADATA_PATH = OUTPUT_DIR / "ablation_metadata.json"
results.to_csv(RESULTS_PATH, index=False)

metadata = {
    "reference_run": str(RUN_DIR),
    "dataset_path": str(DATASET_PATH),
    "model_parameters": CATBOOST_PARAMS,
    "hparams": hparams,
    "split_rows": {
        "train": len(train_frame),
        "validation": len(validation_frame),
        "test": len(test_frame),
    },
    "split_timestamps": {
        "train_end": str(train_frame[TIME_COL].iloc[-1]),
        "validation_start": str(validation_frame[TIME_COL].iloc[0]),
        "validation_end": str(validation_frame[TIME_COL].iloc[-1]),
        "test_start": str(test_frame[TIME_COL].iloc[0]),
        "test_end": str(test_frame[TIME_COL].iloc[-1]),
    },
    "feature_blocks": FEATURE_BLOCKS,
    "threshold_selection": {
        "data": "validation only",
        "false_alert_budget_per_month": FALSE_ALERT_BUDGET,
        "threshold_grid_size": THRESHOLD_GRID_SIZE,
    },
}
with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, default=str)

print(f"Saved metrics: {RESULTS_PATH}")
print(f"Saved metadata: {METADATA_PATH}")

In [ ]:
plot_data = results[results["feature_set"] != "All features"].copy()
plot_metrics = [
    ("delta_test_auc", "Δ ROC AUC"),
    ("delta_test_auprc", "Δ AUPRC"),
    ("delta_test_event_recall", "Δ event recall"),
    ("delta_test_net_utility_per_event", "Δ net utility / event"),
]
fig, axes = plt.subplots(
    1,
    len(plot_metrics),
    figsize=(18, max(4.5, 0.48 * len(plot_data))),
    constrained_layout=True,
)
for ax, (metric, title) in zip(axes, plot_metrics):
    ordered = plot_data.sort_values(metric, ascending=True, na_position="first")
    values = ordered[metric].to_numpy(dtype=float)
    colors = [
        "#d95f02" if np.isfinite(value) and value < 0 else "#1b9e77"
        for value in values
    ]
    ax.barh(ordered["removed_block"], values, color=colors, alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Ablation minus full-feature baseline")
    ax.grid(axis="x", alpha=0.25)

fig.suptitle("Held-out impact of removing each economic feature block", fontsize=14)
plt.show()

## Reading the results

Use the deltas as a block-level sensitivity check for this chronological holdout. A negative change in AUPRC or net utility after removing a block means the full-feature model performed better with that block available. A higher test score after removal can reflect redundancy, sample variation, or overfitting in the full model; it is not a causal estimate of the block's value.

Each model's threshold was selected on validation under the saved false-alert budget. The held-out test period contains a limited number of independent depeg events, so interpret event recall and utility alongside the event count and lead-time columns.